# Step 1 batch runner

## Summary (English)

**PURPOSE**
This notebook is a *batch driver* for the per-trial Step 1 pipeline (`step1_phantom_video_analysis.ipynb`). Step 1 normally processes one trial at a time, controlled by a hard-coded `MAT_FILE = ...` variable. Instead of editing that path by hand and re-running Step 1 for every trial, this runner loops over every `.mat` file in a chosen folder, and for each one it:

1. Loads `step1_phantom_video_analysis.ipynb` into memory (the file on disk is never modified).
2. Rewrites every `MAT_FILE = ...` assignment in the in-memory copy to point at the current trial's `.mat` file.
3. Executes that modified copy in a **fresh Python kernel**, so no variables or state leak between trials.
4. Lets Step 1 write its normal outputs next to the `.mat`, exactly as it would in single-trial mode.

It fits into the project as the "run Step 1 over a whole dataset" convenience layer — the entry point you run once to process an entire folder of trials.

**EXPECTED INPUTS**
- **The Step 1 notebook** — `step1_phantom_video_analysis.ipynb` (`.ipynb`), located at the path set in `STEP1_NB`. It must contain at least one `MAT_FILE = ...` assignment (either the single-line `MAT_FILE = r'...'` form or the multi-line `MAT_FILE = ( r'...' r'...' )` parenthesised form); the runner raises an error if none is found.
- **A folder of trials** — `DATA_DIR`, containing one `.mat` file per trial. Each `.mat` filename is treated as an opaque **base name** (its stem); this runner does not parse or interpret the base name. macOS metadata stubs (`._*.mat`) are ignored.
- **Per-trial companion files** required by Step 1 itself, sitting alongside each `.mat`. Step 1 locates these from the `.mat` base name, so they are expected to be named `<base name> + <suffix>`, e.g. `<base name>_TopCamera.csv` and `<base name>_PhantomCamera.xlsx`. (Their exact contents/columns are Step 1's concern, not this runner's.)

**EXPECTED OUTPUTS**
This runner produces no data files of its own. It triggers Step 1, which writes its usual per-trial outputs into `DATA_DIR` next to each `.mat`, named as `<base name> + <suffix>`:
- `<base name>_phantom_trigger_timedstamp.csv`
- `<base name>_basler_trigger_timestamp.csv`
- `<base name>_combined.csv`
- accompanying `.svg` figures

The only direct output of this notebook is a **printed progress/summary log** (per-trial `[OK]`/`[SKIP]`/`[FAIL]` lines plus a final tally of done / skipped / failed / total time). It returns no value and saves no executed-notebook copy to disk.

**Defaults / behaviour:** skips any trial that already has a `<base name>_combined.csv` (`SKIP_IF_COMBINED_EXISTS`); on a per-trial error it logs the failure and continues to the next trial (`CONTINUE_ON_ERROR`); uses a fresh kernel per trial with a 30-minute timeout (`TIMEOUT_S = 1800`).

**Requirements:** `nbformat` and `nbclient` (both ship with Jupyter).

---

## 摘要（中文）

**用途**
本 notebook 是单试次 Step 1 流程（`step1_phantom_video_analysis.ipynb`）的**批量调度器**。Step 1 本身一次只处理一个试次，由其内部写死的 `MAT_FILE = ...` 变量决定处理哪一个文件。为了避免每个试次都手动改这个路径再重跑一遍 Step 1，本调度器会遍历指定文件夹中的每一个 `.mat` 文件，并对每个文件执行：

1. 把 `step1_phantom_video_analysis.ipynb` 读入内存（磁盘上的原文件**不会被修改**）。
2. 在这份内存副本中，把所有 `MAT_FILE = ...` 赋值改写为指向当前试次的 `.mat` 文件。
3. 在一个**全新的 Python 内核**中执行这份改写后的副本，从而保证试次之间不会有变量或状态残留。
4. 让 Step 1 像单试次模式一样，把它的常规输出写到 `.mat` 文件旁边。

在整个项目中，它扮演"对整个数据集运行 Step 1"的便捷层——只需运行它一次，即可处理一整个文件夹的试次。

**预期输入**
- **Step 1 notebook**——`step1_phantom_video_analysis.ipynb`（`.ipynb`），位于 `STEP1_NB` 指定的路径。它必须至少包含一处 `MAT_FILE = ...` 赋值（既支持单行的 `MAT_FILE = r'...'` 形式，也支持多行括号拼接的 `MAT_FILE = ( r'...' r'...' )` 形式）；若一处都找不到，调度器会报错。
- **试次文件夹**——`DATA_DIR`，其中每个试次对应一个 `.mat` 文件。每个 `.mat` 的文件名被当作一个不透明的**基础名（base name，即不含扩展名的主干）**来处理；本调度器不会解析或解读这个基础名。macOS 的元数据残留文件（`._*.mat`）会被忽略。
- **Step 1 自身所需的每试次配套文件**，与各 `.mat` 放在同一文件夹中。Step 1 依据 `.mat` 的基础名来定位它们，因此它们的命名约定为 `<基础名> + <后缀>`，例如 `<基础名>_TopCamera.csv` 和 `<基础名>_PhantomCamera.xlsx`。（这些文件的具体内容/列由 Step 1 负责，与本调度器无关。）

**预期输出**
本调度器自身不产生任何数据文件。它触发 Step 1，由 Step 1 把常规的每试次输出写入 `DATA_DIR`、紧挨着对应的 `.mat`，命名同样为 `<基础名> + <后缀>`：
- `<基础名>_phantom_trigger_timedstamp.csv`
- `<基础名>_basler_trigger_timestamp.csv`
- `<基础名>_combined.csv`
- 配套的 `.svg` 图

本 notebook 唯一的直接输出是**打印出来的进度/汇总日志**（每个试次的 `[OK]`/`[SKIP]`/`[FAIL]` 行，以及最后的完成 / 跳过 / 失败 / 总耗时统计）。它不返回任何值，也不会把执行后的 notebook 副本存到磁盘。

**默认行为：** 若某试次已存在 `<基础名>_combined.csv`，则跳过该试次（`SKIP_IF_COMBINED_EXISTS`）；某试次出错时记录失败信息并继续处理下一个（`CONTINUE_ON_ERROR`）；每个试次使用全新内核，超时上限为 30 分钟（`TIMEOUT_S = 1800`）。

**依赖：** `nbformat` 与 `nbclient`（二者均随 Jupyter 一起安装）。

### Block 1 — imports & configuration

Imports the standard-library helpers (`os`, `re`, `time`, `traceback`, `pathlib`) plus `nbformat`/`nbclient`, which are the two libraries that let Python read a notebook as data and execute it programmatically. The rest of the cell is the **control panel** for the whole run — every knob you would normally tweak lives here:
- `STEP1_NB` — full path to the Step 1 notebook that will be driven.
- `DATA_DIR` / `RECURSIVE` — which folder of `.mat` trials to process, and whether to descend into sub-folders.
- `SKIP_IF_COMBINED_EXISTS` — skip trials already finished (those that have a `_combined.csv`).
- `CONTINUE_ON_ERROR` — keep going past a failing trial vs. stop and re-raise.
- `TIMEOUT_S` — per-trial kernel timeout (1800 s = 30 min).
- `KERNEL_NAME` — which installed Jupyter kernel to run each trial in.

No work happens yet; this cell only defines values.

---

### 代码块 1 —— 导入与配置

导入标准库工具（`os`、`re`、`time`、`traceback`、`pathlib`），以及 `nbformat`/`nbclient` 这两个关键库——它们让 Python 能把 notebook 当作数据读入并以编程方式执行。本单元其余部分是整个运行过程的**控制面板**，所有你通常需要调整的参数都集中在这里：
- `STEP1_NB` —— 被调度的 Step 1 notebook 的完整路径。
- `DATA_DIR` / `RECURSIVE` —— 要处理哪个 `.mat` 试次文件夹，以及是否递归进入子文件夹。
- `SKIP_IF_COMBINED_EXISTS` —— 跳过已完成的试次（即已存在 `_combined.csv` 的试次）。
- `CONTINUE_ON_ERROR` —— 某试次失败时是继续往下跑，还是停下并抛出异常。
- `TIMEOUT_S` —— 每个试次的内核超时上限（1800 秒 = 30 分钟）。
- `KERNEL_NAME` —— 用哪个已安装的 Jupyter 内核来运行每个试次。

本单元尚不执行任何实际工作，只负责定义这些参数值。

In [4]:
# Block 1 — imports + parameters
import os
import re
import time
import traceback
from pathlib import Path

import nbformat
from nbclient import NotebookClient

# Path to the Step 1 notebook (the per-trial pipeline)
STEP1_NB = Path(
    r"h:\.shortcut-targets-by-id\10pxdlRXtzFB-abwDGi0jOGOFFNm3pmFK"
    r"\Tuthill Lab Shared\Yichen\Spiracle\Analysis Code"
    r"\step1_phantom_video_analysis.ipynb"
)

# Folder of trials to process
DATA_DIR  = Path(r"C:\Users\Lylah\Desktop\data_processing\SpINB_ChR\SpINB_ChR_3s")
RECURSIVE = False

# Skip a trial if <stem>_combined.csv already exists (set False to force-rerun)
SKIP_IF_COMBINED_EXISTS = True

# On per-trial error: skip and continue (False = stop and re-raise)
CONTINUE_ON_ERROR = True

# Kernel timeout per trial (seconds)
TIMEOUT_S = 1800

# Kernel name (must match a kernel installed in this Jupyter)
KERNEL_NAME = "python3"

### Block 2 — helper functions

Defines the three workhorse functions (nothing runs yet — these are just definitions used by Block 3):

- **Two regular expressions** (`_MAT_FILE_PARENS`, `_MAT_FILE_LINE`) that recognise the two ways Step 1 might write its `MAT_FILE = ...` line: the multi-line parenthesised form and the plain single-line form.
- **`find_mat_files(data_dir, recursive)`** — returns a sorted list of the `.mat` files to process, skipping macOS `._*` metadata stubs.
- **`override_mat_file_in_nb(nb, mat_path)`** — rewrites every `MAT_FILE = ...` assignment in an in-memory notebook so it points at `mat_path`, and returns how many cells were changed. Two deliberate safeguards are worth noting: it writes the path with **forward slashes inside a raw string** so Windows backslashes can never be misinterpreted as escape sequences, and it passes a **function** to `re.sub` (rather than a replacement string) so backslashes in the path are never treated as regex backreferences.
- **`run_step1_on(mat_path, step1_nb_path, ...)`** — the per-trial entry point: reads Step 1 fresh from disk, applies the override, raises if no `MAT_FILE` was found, then runs the notebook in a fresh kernel (with the working directory set to Step 1's folder so its relative paths resolve, and `allow_errors=False` so any cell error stops that trial).

---

### 代码块 2 —— 辅助函数

定义三个核心工具函数（此处仍不执行任何操作，只是供代码块 3 调用的定义）：

- **两个正则表达式**（`_MAT_FILE_PARENS`、`_MAT_FILE_LINE`），用于识别 Step 1 可能采用的两种 `MAT_FILE = ...` 写法：多行括号拼接形式，以及普通单行形式。
- **`find_mat_files(data_dir, recursive)`** —— 返回待处理 `.mat` 文件的排序列表，并跳过 macOS 的 `._*` 元数据残留文件。
- **`override_mat_file_in_nb(nb, mat_path)`** —— 把内存中 notebook 的每一处 `MAT_FILE = ...` 赋值改写为指向 `mat_path`，并返回被修改的单元数。其中有两处刻意的防护值得注意：它用**原始字符串（raw string）配合正斜杠**来写路径，以确保 Windows 的反斜杠绝不会被误读为转义序列；并且它向 `re.sub` 传入一个**函数**（而非替换字符串），从而保证路径中的反斜杠绝不会被当作正则的反向引用。
- **`run_step1_on(mat_path, step1_nb_path, ...)`** —— 单试次的入口函数：从磁盘重新读取 Step 1，应用上面的改写，若未找到任何 `MAT_FILE` 则报错，然后在全新内核中运行该 notebook（工作目录设为 Step 1 所在文件夹，以便其相对路径能正确解析；并设 `allow_errors=False`，使任一单元出错都会中止该试次）。

In [5]:
# Block 2 — helpers: discover MATs, rewrite MAT_FILE, run one trial

# Match the two MAT_FILE assignment styles Step 1 currently uses:
#   MAT_FILE = ( r'...' r'...' )         (multi-line concatenation in parens)
#   MAT_FILE = r'...'  or  "..."         (single-line)
_MAT_FILE_PARENS = re.compile(r"^MAT_FILE\s*=\s*\([^)]*\)", re.MULTILINE | re.DOTALL)
_MAT_FILE_LINE   = re.compile(r"^MAT_FILE\s*=\s*[rRuU]?['\"][^'\"]*['\"]", re.MULTILINE)


def find_mat_files(data_dir: Path, recursive: bool = False):
    """Return sorted list of *.mat under data_dir, excluding macOS dot-files and *_*.mat helper files."""
    it = data_dir.rglob("*.mat") if recursive else data_dir.glob("*.mat")
    out = []
    for p in sorted(it):
        if p.name.startswith("._"):  # macOS metadata stub
            continue
        out.append(p)
    return out


def override_mat_file_in_nb(nb, mat_path: Path) -> int:
    """Replace every MAT_FILE = ... assignment in the notebook so it points at mat_path.

    Returns the number of cells modified.

    Two subtle requirements:
      1. Emit a forward-slash + raw-string form (`MAT_FILE = r"C:/.../foo.mat"`)
         so Windows backslashes can never be misread as Python string escapes.
      2. Pass a function to re.sub. re.sub treats backslashes in a string
         replacement as backreferences, which would mangle Windows paths.
    """
    posix_path = mat_path.as_posix()  # forward slashes; works on Windows scipy/os
    new_assign = f'MAT_FILE = r"{posix_path}"'
    repl = lambda _m: new_assign  # function form -> replacement is literal

    n_cells_modified = 0
    for cell in nb.cells:
        if cell.cell_type != "code":
            continue
        src = cell.source
        new_src = src
        if _MAT_FILE_PARENS.search(new_src):
            new_src = _MAT_FILE_PARENS.sub(repl, new_src)
        if _MAT_FILE_LINE.search(new_src):
            new_src = _MAT_FILE_LINE.sub(repl, new_src)
        if new_src != src:
            cell.source = new_src
            n_cells_modified += 1
    return n_cells_modified


def run_step1_on(mat_path: Path, step1_nb_path: Path,
                 timeout_s: int = 1800, kernel_name: str = "python3") -> int:
    """Execute Step 1 once with MAT_FILE pointed at mat_path. Returns # cells whose MAT_FILE was overridden."""
    nb = nbformat.read(str(step1_nb_path), as_version=4)
    n_subs = override_mat_file_in_nb(nb, mat_path)
    if n_subs == 0:
        raise RuntimeError(
            f"Found no MAT_FILE assignment in {step1_nb_path.name} — has the notebook been refactored?"
        )

    client = NotebookClient(
        nb,
        timeout=timeout_s,
        kernel_name=kernel_name,
        # Run with cwd = the notebook's folder so any relative imports resolve as in interactive use
        resources={"metadata": {"path": str(step1_nb_path.parent)}},
        # Don't write the executed notebook to disk
        allow_errors=False,
    )
    client.execute()
    return n_subs

### Block 3 — the runner (the cell you actually execute)

This is the only cell that does work. It first sanity-checks that both `STEP1_NB` and `DATA_DIR` exist (raising immediately if not), then gathers the list of `.mat` trials and prints them. It loops over every trial, and for each one:

1. Computes the expected `<base name>_combined.csv` and, if `SKIP_IF_COMBINED_EXISTS` is on and that file already exists, logs `[SKIP]` and moves on.
2. Otherwise calls `run_step1_on(...)`, timing it; on success logs `[OK]` with the elapsed time, on exception logs `[FAIL]` and records the error.
3. Honours `CONTINUE_ON_ERROR` — if that flag is off, the first failure re-raises and stops the whole batch.

It keeps running counts (`n_done`, `n_skip`, `n_fail`) and a `failures` list, and at the end prints a **summary**: how many trials were done / skipped / failed, the total wall-clock time, and a one-line excerpt of each failure's error message. This printed log is the notebook's only output.

---

### 代码块 3 —— 运行器（你真正要执行的单元）

这是唯一执行实际工作的单元。它首先检查 `STEP1_NB` 和 `DATA_DIR` 是否都存在（不存在则立即报错），然后收集 `.mat` 试次列表并打印出来。接着它遍历每一个试次，对每个试次执行：

1. 计算出预期的 `<基础名>_combined.csv`；若开启了 `SKIP_IF_COMBINED_EXISTS` 且该文件已存在，则打印 `[SKIP]` 并跳到下一个。
2. 否则调用 `run_step1_on(...)` 并计时；成功则打印 `[OK]` 及耗时，抛出异常则打印 `[FAIL]` 并记录错误。
3. 遵循 `CONTINUE_ON_ERROR`：若该开关关闭，则首次失败就会重新抛出异常并中止整个批处理。

它维护运行计数（`n_done`、`n_skip`、`n_fail`）和一个 `failures` 列表，并在最后打印**汇总**：完成 / 跳过 / 失败的试次数量、总墙钟耗时，以及每个失败错误信息的单行摘要。这段打印日志是本 notebook 唯一的输出。

In [6]:
# Block 3 — runner

if not STEP1_NB.exists():
    raise FileNotFoundError(f"Step 1 notebook not found at: {STEP1_NB}")
if not DATA_DIR.exists():
    raise FileNotFoundError(f"DATA_DIR not found: {DATA_DIR}")

mat_files = find_mat_files(DATA_DIR, recursive=RECURSIVE)
print(f"Found {len(mat_files)} MAT file(s) under {DATA_DIR}")
for p in mat_files:
    print(f"  {p.name}")

n_done, n_skip, n_fail = 0, 0, 0
failures = []
t_total0 = time.time()

for i, mat in enumerate(mat_files, 1):
    combined_csv = mat.with_name(mat.stem + "_combined.csv")

    print(f"\n=== [{i}/{len(mat_files)}] {mat.name} ===")
    if SKIP_IF_COMBINED_EXISTS and combined_csv.exists():
        print(f"  [SKIP] {combined_csv.name} already exists")
        n_skip += 1
        continue

    t0 = time.time()
    try:
        n_subs = run_step1_on(mat, STEP1_NB, TIMEOUT_S, KERNEL_NAME)
        dt = time.time() - t0
        print(f"  [OK] {n_subs} MAT_FILE override(s); {dt:.1f} s; produced {combined_csv.name}")
        n_done += 1
    except Exception as e:
        n_fail += 1
        dt = time.time() - t0
        print(f"  [FAIL] {dt:.1f} s; {type(e).__name__}: {e}")
        failures.append((mat.name, type(e).__name__, str(e)))
        if not CONTINUE_ON_ERROR:
            raise

dt_total = time.time() - t_total0
print(f"\n=== Summary ===")
print(f"  done    : {n_done}")
print(f"  skipped : {n_skip}")
print(f"  failed  : {n_fail}")
print(f"  total   : {dt_total:.1f} s")
if failures:
    print("\nFailures:")
    for name, etype, emsg in failures:
        first_line = (emsg.splitlines() or [""])[0][:200]
        print(f"  - {name}  [{etype}]  {first_line}")

Found 8 MAT file(s) under C:\Users\Lylah\Desktop\data_processing\SpINB_ChR\SpINB_ChR_3s
  2026_0507_095718_SpINB_ChR_IS46338_ChR_4d_F_Fly1_Trial1_3000ms_thorax_sp1_ON.mat
  2026_0507_102707_SpINB_ChR_IS46338_ChR_4d_F_Fly2_Trial2_3000ms_thorax_sp1_ON.mat
  2026_0507_103610_SpINB_ChR_IS46338_ChR_4d_F_Fly3_Trial2_3000ms_thorax_sp1_ON.mat
  2026_0507_111119_SpINB_ChR_IS46338_ChR_4d_F_Fly4_Trial2_3000ms_thorax_sp1_ON.mat
  2026_0507_112119_SpINB_ChR_IS46338_ChR_4d_F_Fly5_Trial1_3000ms_thorax_sp1_ON.mat
  2026_0507_112819_SpINB_ChR_IS46338_ChR_4d_F_Fly6_Trial1_3000ms_thorax_sp1_ON.mat
  2026_0507_113419_SpINB_ChR_IS46338_ChR_4d_F_Fly7_Trial1_3000ms_thorax_sp1_ON.mat
  2026_0507_121238_SpINB_ChR_IS46338_ChR_4d_F_Fly8_Trial1_3000ms_thorax_sp1_ON.mat

=== [1/8] 2026_0507_095718_SpINB_ChR_IS46338_ChR_4d_F_Fly1_Trial1_3000ms_thorax_sp1_ON.mat ===
  [OK] 1 MAT_FILE override(s); 23.4 s; produced 2026_0507_095718_SpINB_ChR_IS46338_ChR_4d_F_Fly1_Trial1_3000ms_thorax_sp1_ON_combined.csv

=== [2/8] 202